# Visualizing the *A. hallii* & *B. infantis* Community Network

This notebook simulates this community and extracts the metabolic network graph (and relevant node/edge attributes) to then visual in cytoscape.


In [1]:
import gifba
import cobra as cb
import xml.etree.ElementTree as ET
import re
import networkx as nx

In [2]:
# AGORA models
AH_PATH = "data/Eubacterium_hallii_DSM_3353.mat"
BI_PATH = "data/Bifidobacterium_longum_infantis_ATCC_15697.mat"
SBML_PATH = "data/2_ahallii_binfantis_metabolism.sbml"
GRAPHML_PATH = "data/2_ahallii_binfantis_metabolism.xml"

## Greedy Interaction FBA

In [3]:
AH = cb.io.load_matlab_model(AH_PATH)
BI = cb.io.load_matlab_model(BI_PATH)

models = [BI, AH]
ids_list = ["BI", "AH"]
rel_abund = [0.5, 0.5]

media = {
	'EX_o2(e)': 0, #aerobic/anaerobic
	'EX_h2o(e)': -1000,
	'EX_pi(e)': -1000,
	'EX_fe2(e)': -1000,
	'EX_fe3(e)': -1000,
	'EX_zn2(e)': -1000,
	'EX_so4(e)': -1000,
	'EX_cu2(e)': -1000,
	'EX_k(e)': -1000,
	'EX_mg2(e)': -1000,
	'EX_mn2(e)': -1000,
	'EX_cd2(e)': -1000,
	'EX_cl(e)': -1000,
	'EX_ca2(e)': -1000,
	'EX_cobalt2(e)': -1000,
	'EX_glc_D(e)': -10,
	'EX_nh4(e)': -20,

	'EX_ribflv(e)': -1000,
	'EX_pnto_R(e)': -1000,
	'EX_nac(e)': -1000,
	'EX_his_L(e)': -1000,
	'EX_asn_L(e)': -1000,
	'EX_glycys(e)': -1000,

	'EX_lys_L(e)': -1000,
	'EX_ala_L(e)': -1000,
	'EX_met_L(e)': -1000,
	'EX_leu_L(e)': -1000,
	'EX_hxan(e)': -1000,    
    'EX_glyglu(e)': -1000
}

Set parameter Username
Set parameter LicenseID to value 2773321
Academic license - for non-commercial use only - expires 2027-02-01


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


In [4]:
# initialize community
community = gifba.gifbaObject(models, 
                              media,
                              rel_abund=rel_abund) 

# run iterations
media_flux, org_flux = community.run_gifba(iters=25, 
                                           method="pfba", 
                                           v=False)

Read LP format model from file /tmp/tmpfdilgjwv.lp
Reading time = 0.00 seconds
: 932 rows, 2064 columns, 8440 nonzeros
Read LP format model from file /tmp/tmpv4948192.lp
Reading time = 0.00 seconds
: 980 rows, 2102 columns, 8980 nonzeros


In [5]:
# compartmentalize the community model
cfba_model, micom_comm, objective_dict = gifba.utils.prep_micom_cfba("compartmentalized_model", ["AH", "BI"], [AH_PATH, BI_PATH], rel_abund=rel_abund)

Output()

No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.

Using regular expression found the following compartments:c, e

No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.

Using regular expression found the following compartments:c, e

### Generate compartmentalized network visualization

In [6]:
# associate gifba organism fluxes with SBML model
# BI = model idx 0, AH = model idx 1
flux_dict = {}

for rxn in cfba_model.reactions:
    # fix bug with reversible rxns
    rxn.lower_bound = float(rxn.lower_bound)
    rxn.upper_bound = float(rxn.upper_bound)

cb.io.write_sbml_model(cfba_model, SBML_PATH)

In [13]:
def sbml_to_graphml(sbml_filepath, mdl_ids, org_flux_df, env_flux_df, output_graphml_path="community_network.graphml"):
    """
    Parses an SBML microbial community model, extracts flux annotations from 
    reaction notes, computes node and edge flux metrics, and saves a Cytoscape-ready GraphML file.
    """
    tree = ET.parse(sbml_filepath)
    root = tree.getroot()

    # Automatically handle SBML XML namespaces
    ns = {'sbml': root.tag.split('}')[0].strip('{')} if '}' in root.tag else {}

    def get_elements(parent, tag):
        return parent.findall(f".//sbml:{tag}" if ns else f".//{tag}", ns)

    G = nx.DiGraph()

    # 1. Parse Metabolites (Species)
    metabolite_ids = set()
    for sp in get_elements(root, 'species'):
        sp_id = sp.attrib.get('id')
        sp_name = sp.attrib.get('name', sp_id)
        compartment = sp.attrib.get('compartment', 'default')
        metabolite_ids.add(sp_id)
        org_id = None
        org_idx = None
        for i, mdl_id in enumerate(mdl_ids):
            if sp_id.endswith(f"__{mdl_id}"):
                org_id = mdl_id
                org_idx = i
                break
        if org_id is None:
            org_id = "m"  # default to "m" for media if no organism ID is found

        G.add_node(
            sp_id,
            label=sp_name,
            node_type="metabolite",
            node_type_and_id="metabolite__" + org_id,
            compartment=compartment,
            node_flux=0.0, # magnitude
            raw_flux=0.0, # raw flux
            active=False
        )

    # 2. Parse Reactions and Flux Notes
    for rxn in get_elements(root, 'reaction'):
        rxn_id = rxn.attrib.get('id')
        rxn_name = rxn.attrib.get('name', rxn_id)
        compartment = rxn.attrib.get('compartment', 'default')
        if compartment != "default":
            print(rxn_id, rxn_name, compartment)

        # get org_id for the rxn, based on the suffix of the rxn_id and the provided list of model IDs
        org_id = None
        org_idx = None
        for i, mdl_id in enumerate(mdl_ids):
            if rxn_id.endswith(f"__{mdl_id}"):
                org_id = mdl_id
                org_idx = i
                break
        org_id = "m" if org_id is None else org_id  # default to "m" for media if no organism ID is found
        rxn_orig_id = rxn_id.replace("__40__", "(").replace("__41__", ")").replace("__91__", "[").replace("__93__", "]")
        rxn_orig_id = rxn_orig_id.replace(f"__{org_id}", "") if org_id else rxn_orig_id
        rxn_orig_id = rxn_orig_id[2:] if rxn_orig_id.startswith("R_") else rxn_orig_id

        # replace "DM_biomass" with "EX_biomass" in the reaction ID if present
        if "DM_biomass" in rxn_orig_id:
            rxn_orig_id = rxn_orig_id.replace("DM_biomass", "EX_biomass")

        # address exchange rxns
        rxn_orig_id = rxn_orig_id[:-2] + "(e)" if rxn_orig_id.endswith("_m") else rxn_orig_id


        # Extract numerical flux value from org_flux/media_flux DFs
        raw_flux = 0.0
        if org_id is not None and org_idx is not None:
            raw_flux = org_flux_df.loc[org_idx][rxn_orig_id]
        else:
            raw_flux = org_flux_df[rxn_orig_id].sum()
        rxn_flux_mag = abs(raw_flux)

        G.add_node(
            rxn_id,
            label=rxn_name,
            node_type="reaction",
            node_type_and_id="reaction__" + org_id,
            compartment=compartment,
            node_flux=rxn_flux_mag,
            raw_flux=raw_flux,
            active=bool(rxn_flux_mag > 0.0)
        )

        # Process Substrates (Metabolite -> Reaction)
        reactants = rxn.find('sbml:listOfReactants' if ns else 'listOfReactants', ns)
        if reactants is not None:
            for sr in reactants.findall('sbml:speciesReference' if ns else 'speciesReference', ns):
                sp_id = sr.attrib.get('species')
                stoich = float(sr.attrib.get('stoichiometry', 1.0))
                edge_flux_raw = raw_flux * stoich

                G.add_edge(
                    sp_id,
                    rxn_id,
                    interaction="substrate",
                    stoichiometry=stoich,
                    edge_flux=abs(edge_flux_raw),
                    raw_flux=edge_flux_raw,
                    reverse=bool(edge_flux_raw < 0)
                )

                # Accumulate flux into metabolite pool
                if sp_id in G.nodes:
                    G.nodes[sp_id]['raw_flux'] += edge_flux_raw
                    G.nodes[sp_id]['node_flux'] = abs(G.nodes[sp_id]['raw_flux'])
                    G.nodes[sp_id]['active'] = bool(G.nodes[sp_id]['node_flux'] > 0.0)

        # Process Products (Reaction -> Metabolite)
        products = rxn.find('sbml:listOfProducts' if ns else 'listOfProducts', ns)
        if products is not None:
            for pr in products.findall('sbml:speciesReference' if ns else 'speciesReference', ns):
                sp_id = pr.attrib.get('species')
                stoich = float(pr.attrib.get('stoichiometry', 1.0))
                edge_flux_raw = raw_flux * stoich

                G.add_edge(
                    rxn_id,
                    sp_id,
                    interaction="product",
                    stoichiometry=stoich,
                    edge_flux=abs(edge_flux_raw),
                    raw_flux=edge_flux_raw,
                    reverse=bool(edge_flux_raw < 0)
                )

                # Accumulate flux into metabolite pool
                if sp_id in G.nodes:
                    G.nodes[sp_id]['raw_flux'] += edge_flux_raw
                    G.nodes[sp_id]['node_flux'] = abs(G.nodes[sp_id]['raw_flux'])
                    G.nodes[sp_id]['active'] = bool(G.nodes[sp_id]['node_flux'] > 0.0)

    # Write out to GraphML file
    nx.write_graphml(G, output_graphml_path)
    print(f"GraphML successfully created at: {output_graphml_path}")
    print(f"Total Nodes: {G.number_of_nodes()} | Total Edges: {G.number_of_edges()}")



In [14]:
# Update 'model.xml' to your SBML file path
sbml_to_graphml(SBML_PATH, ["AH", "BI"], org_flux, media_flux, GRAPHML_PATH)

GraphML successfully created at: data/2_ahallii_binfantis_metabolism.xml
Total Nodes: 4369 | Total Edges: 9178
